In [ ]:
%pip install pandas numpy scikit-learn matplotlib

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [ ]:
train_df=pd.read_csv("kepler_train_by_host.csv");
test_df=pd.read_csv("kepler_test_by_host.csv");
print(train_df.shape);
print(test_df.shape);

In [ ]:
print("Training target Distribution:")
print(train_df["koi_disposition"].value_counts())
print("\nTesting target Distribution:")
print(test_df["koi_disposition"].value_counts())


In [ ]:
train_known=train_df[train_df["koi_disposition"].isin(["CONFIRMED","FALSE POSITIVE"])].copy()
test_known=test_df[test_df["koi_disposition"].isin(["CONFIRMED","FALSE POSITIVE"])].copy()
candidate_df=test_df[test_df["koi_disposition"]=="CANDIDATE"].copy()
print("Known training objects",len(train_known))
print("Known testing objects",len(test_known))
print("Candidate objects",len(candidate_df))

In [ ]:
train_known["target"]=(train_known["koi_disposition"]=="CONFIRMED").astype(int)
test_known["target"]=(test_known["koi_disposition"]=="CONFIRMED").astype(int)
print(train_known["target"].value_counts())

In [ ]:
feature_columns=[
    "koi_period",
    "koi_duration",
    "koi_dept",
    "koi_prad",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "koi_srad",
    "koi_kepmag"
]
available_features=[ 
    col for col in feature_columns
    if col in train_known.columns
]
missing_features=[
    col for col in feature_columns
    if col not in train_known.columns
]

print("Features used:")
print(available_features)

print("\nMissing features:")
print(missing_features)

X_train=train_known[available_features].copy()
Y_train=train_known["target"].copy()

X_test=test_known[available_features].copy()
Y_test=test_known["target"].copy()

X_candidates=candidate_df[available_features].copy()

print("\n X_train shape:",X_train.shape)
print("X_test shape:",X_test.shape)
print("X_candidates shape:",X_candidates.shape)

In [ ]:
logistic_model=Pipeline([
    ("scaler",StandardScaler()),
    ("model",LogisticRegression(max_inter=1000,random_state=42))
])

logistic_model.fir(X_train,Y_train)

print("Logistic regression model trained successfully")

In [ ]:
logistic_pred=logistic_model.predict(X_test)
logistic_prob=logistic_model.predict_proba(X_test)[:,1]

print("Logistic Regression Results")
print("---------------------------")

print("Accuracy:",accuracy_score(Y_test,logistic_pred))
print("Precision:",precision_score(Y_test,logistic_pred,zero_division=0))
print("Recall:",recall_score(Y_test,logistic_pred,zero_division=0))
print("F1 Score:",f1_score(Y_test,logistic_pred,zero_division=0))
print("ROC-AUC:",roc_auc_score(Y_test,logistic_prob))

print("\nClassification Report:")
print(classification_report(
    Y_test,
    logistic_pred,
    target_names=["FALSE POSITIVE, "CONFIRMED],
    zero_division=0
))

In [ ]:
rf_model=RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train,Y_train)

print("Random Forest model trained successfully")

In [ ]:
rf_pred=rf_model.predict(X_test)
rf_prob=rf_model.predict_proba(X_test)[:,1]

print("Random Forest Results")
print("---------------------")

print("Accuracy: ",accuracy_score(Y_test,rf_pred))
print("Precision: ",precision_score(Y_test,rf_pred,zero_division=0))
print("Recall: ",recall_score(Y_test,rf_pred,zero_division=0))
print("F1 Score: ",f1_score(Y_test,rf_pred,zero_division=0))
print("ROC-AUC: ",roc_auc_score(Y_test,rf_prob))

print("\n Classification Report: ")
print(classification_report(
    Y_test,
    rf_pred,
    target_names=["FALSE POSITIVE", "CONFIRMED"],
    zero_division=0
))

In [ ]:
model_comparison=pd.DataFrame({
    "Model":["Logistic Regression", "Random Forest"],
    "Accuracy":[
        accuracy_score(Y_test, logistic_pred),
        accuracy_score(Y_test,rf_pred)
    ],
    "Precision":[
        precision_score(Y_test,logistic_pred,zero_division=0),
        precision_score(Y_test,rf_pred,zero_division=0)
    ],
    "Recall":[
        recall_score(Y_test,logistic_pred,zero_division=0),
        recall_score(Y_test,rf_pred,zero_division=0)
    ],
    "F1 Score":[
        f1_score(Y_test,logistic_pred,zero_division=0),
        f1_score(Y_test,rf_pred,zero_division=0)
    ],
    "ROC-AUC":[
        roc_auc_score(Y_test,logistic_prob),
        roc_auc_score(Y_test,rf_prob)
    ]
})

print(model_comparison)

In [ ]:
fig,ax=plt.subplot()

ConfusionMatrixDisplay.from_predictions(
    Y_test,
    rf_pred,
    display_labels=["FALSE POSITIVE","CONFIRMED"],
    ax=ax
)

ax.set_title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
plt.figure()

RocCurveDisplay.from_predictions(
    Y_test,
    logistic_prob,
    name="Logistic Regression"
)

RocCurveDisplay.from_predictions(
    Y_test,
    rf_prob,
    name=-"Random Forest"
)

plt.title("ROC Curve Comparison")
plt.show()

In [ ]:
feature_importance=pd.DataFrame({
    "Feature":available_features,
    "Importance":rf_model.feature_importances_
})

feature_importance=feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

In [ ]:
plt.figure(figsize=(10,6))
plt.barh(
    feature_importance["Feature"],
    feature_importance["Importance"]
)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()

plt.show()

In [ ]:
candidate_prob=rf_model.predict_proba(X_candidates)[:,1]
candidate_results=candidate_df.copy()
candidate_results["predicted_confirmed_probability"]=candidate_prob

print(candidate_results[
    ["kepid","kepoi_name","predicted_confirmed_probability"]
].head(10)
)

In [ ]:
candidate_results=candidate_results.sort_values(
    by="predicted_confirmed_probability",
    ascending=False
).reset_index(drop=True)
candidate_results["priority"]=pd.cut(
    candidate_results["predicted_confirmed_probability"],
    bins=[-np.inf,0.50,0.80,np.inf],
    labels=["Low","Medium","High"]
)
print(
    candidate_results[
    ["kepid","kepoi_name",
     "predicted_confirmed_probability","priority"]
    ].head(20)
)

In [ ]:
candidate_results.to_csv(
    "memeber2_candidate_predictions.csv",
    index=False
)

model_comparison.to_csv(
    "member2_model_comparison.csv",
    index=False
)

feature_importance.to_csv(
    "member2_feature_importance.csv",
    index=False
)

print("All member 2 results saved successfully")